# Ogham OCR: PARSeq Baseline

Fine-tune PARSeq (Permuted Autoregressive Sequence model) for Ogham OCR.

**Architecture:** ViT encoder + Transformer decoder with permutation language modelling (ECCV 2022)

**Why PARSeq:**
- Modern scene text recognition model (~24M params)
- Native custom charset support (no tokenizer surgery needed)
- Smaller than TrOCR-small (24M vs 62M) but uses attention (unlike CTC)
- Three decode modes: autoregressive, non-autoregressive, iterative refinement

**Dataset:** Same 200k synthetic Ogham images from HF Hub.

**Requirements:** Colab with GPU.

---
## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Install PARSeq and dependencies
!pip install -q datasets huggingface_hub editdistance Pillow tqdm torchvision
!pip install -q timm pytorch_lightning

# Clone PARSeq repo for model loading
import os
if not os.path.exists('/content/parseq'):
    !git clone https://github.com/baudm/parseq.git /content/parseq
    !cd /content/parseq && pip install -q -r requirements/core.gpu.txt -e '.[train,test]'
else:
    print('PARSeq already cloned')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ogham_ocr'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints/parseq'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints: {CHECKPOINT_DIR}')

In [ ]:
# Clone ogham repo
import os
REPO_DIR = '/content/ogham_ocr'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/newsyd04/ogham-masters.git {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
from huggingface_hub import login
login()

---
## 2. Configuration

In [ ]:
HF_DATASET_NAME = 'DaraTraining/ogham-synthetic-200k'

EPOCHS = 20
BATCH_SIZE = 64
LR = 1e-4
IMG_SIZE = (32, 128)    # PARSeq default: height x width
MAX_LABEL_LEN = 25      # PARSeq default max sequence length
NUM_WORKERS = 4

# Ogham charset for PARSeq
OGHAM_CHARS = ''.join(chr(c) for c in range(0x1681, 0x169D))  # 28 Ogham letters
CHARSET = OGHAM_CHARS + ' '  # Add space

print(f'Config: {EPOCHS} epochs, batch {BATCH_SIZE}, lr {LR}')
print(f'Image size: {IMG_SIZE}')
print(f'Charset: {len(CHARSET)} chars ({len(OGHAM_CHARS)} Ogham + space)')
print(f'Charset: {CHARSET}')

---
## 3. Load PARSeq Model

In [ ]:
import torch
import sys
sys.path.insert(0, '/content/parseq')

# Load pretrained PARSeq
parseq = torch.hub.load('baudm/parseq', 'parseq', pretrained=True)

print(f'PARSeq loaded: {sum(p.numel() for p in parseq.parameters()):,} params')
print(f'Encoder: {sum(p.numel() for p in parseq.model.encoder.parameters()):,} params')
print(f'Decoder: {sum(p.numel() for p in parseq.model.decoder.parameters()):,} params')
print(f'Head: {parseq.model.head}')
print(f'Original charset: {parseq.hparams.charset_train[:40]}...')
print(f'Image size: {parseq.hparams.img_size}')

In [ ]:
# Rebuild PARSeq with Ogham charset
import torch
import torch.nn as nn

# PARSeq architecture params (from pretrained)
# encoder: ViT with 12 blocks, 384 embed_dim, 6 heads
# decoder: 1-layer transformer decoder

# Strategy: keep pretrained encoder, rebuild head + text_embed for Ogham
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = parseq  # start from pretrained
model_inner = model.model

# Resize output head: 95 (original) -> len(CHARSET) + 2 (BOS + EOS)
# PARSeq uses: [EOS, *charset_chars, BOS, PAD]
num_classes = len(CHARSET) + 3  # charset + EOS + BOS + PAD
old_head = model_inner.head
new_head = nn.Linear(old_head.in_features, num_classes, bias=True).to(device)
nn.init.xavier_uniform_(new_head.weight)
nn.init.zeros_(new_head.bias)
model_inner.head = new_head

# Resize text embeddings
old_embed = model_inner.text_embed
new_embed_dim = old_embed.embedding.weight.shape[1]  # 384
new_text_embed = type(old_embed)(num_classes, new_embed_dim).to(device)
model_inner.text_embed = new_text_embed

# Update tokenizer with Ogham charset
from strhub.data.utils import Tokenizer
model.tokenizer = Tokenizer(CHARSET)

# Update hparams
model.hparams.charset_train = CHARSET
model.hparams.charset_test = CHARSET

model = model.to(device).train()

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Rebuilt model: {total_params:,} params')
print(f'New head: {model_inner.head}')
print(f'New charset size: {num_classes} (28 Ogham + space + EOS + BOS + PAD)')
print(f'Tokenizer BOS={model.tokenizer.bos_id}, EOS={model.tokenizer.eos_id}, PAD={model.tokenizer.pad_id}')

# Quick test
test_encoded = model.tokenizer.encode('ᚐᚌᚊᚊ', device)
print(f'Test encode "ᚐᚌᚊᚊ": {test_encoded}')

---
## 4. Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torchvision import transforms
from PIL import Image
import editdistance

class PARSeqDataset(Dataset):
    """Dataset wrapper for PARSeq training."""

    def __init__(self, hf_dataset, transform, max_label_len=25):
        self.ds = hf_dataset
        self.transform = transform
        self.max_label_len = max_label_len

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]
        try:
            image = sample['image'].convert('RGB')
        except Exception:
            return self.__getitem__((idx + 1) % len(self))

        image = self.transform(image)
        text = sample['ogham_text'].replace('\u1680', ' ')
        # Truncate to max label length
        text = text[:self.max_label_len]

        return image, text

# PARSeq image transforms
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(0.5, 0.5),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(0.5, 0.5),
])

print('Dataset class ready')

In [ ]:
# Load HF dataset
print(f'Loading: {HF_DATASET_NAME}')
hf_train = load_dataset(HF_DATASET_NAME, split='train', token=True)
hf_val = load_dataset(HF_DATASET_NAME, split='validation', token=True)
print(f'Train: {len(hf_train)}, Val: {len(hf_val)}')

train_dataset = PARSeqDataset(hf_train, train_transform, MAX_LABEL_LEN)
val_dataset = PARSeqDataset(hf_val, val_transform, MAX_LABEL_LEN)

def collate_fn(batch):
    images, texts = zip(*batch)
    images = torch.stack(images)
    return images, list(texts)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn,
    persistent_workers=True,
)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

---
## 5. Training

In [ ]:
import time
import json
import torch.nn.functional as F
from tqdm import tqdm

def compute_cer(predictions, references):
    total_dist = 0
    total_chars = 0
    for pred, ref in zip(predictions, references):
        total_dist += editdistance.eval(pred, ref)
        total_chars += max(len(ref), 1)
    return total_dist / total_chars if total_chars > 0 else 0.0

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')

history = {'train_loss': [], 'val_loss': [], 'val_cer': [], 'val_exact_match': []}
best_cer = float('inf')

history_path = f'{CHECKPOINT_DIR}/history.json'
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    print(f'Loaded history: {len(history["train_loss"])} prior epochs')
    best_cer = min(history['val_cer']) if history['val_cer'] else float('inf')

start_epoch = len(history['train_loss'])

print(f'\n{"="*60}')
print(f'Training PARSeq: {EPOCHS} epochs, batch {BATCH_SIZE}, lr {LR}')
print(f'{"="*60}\n')

for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()

    # --- Train ---
    model.train()
    train_loss = 0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f'  Epoch {epoch+1}/{EPOCHS} Train', leave=False)
    for images, texts in pbar:
        images = images.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            logits, loss, loss_numel = model.forward_logits_loss(images, texts)

        if torch.isfinite(loss):
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            n_batches += 1

        pbar.set_postfix(loss=f'{train_loss / max(n_batches, 1):.4f}')

    scheduler.step()
    avg_train_loss = train_loss / max(n_batches, 1)

    # --- Validate ---
    model.eval()
    val_loss = 0
    all_preds = []
    all_refs = []
    n_val = 0

    with torch.no_grad():
        for images, texts in tqdm(val_loader, desc=f'  Epoch {epoch+1}/{EPOCHS} Val', leave=False):
            images = images.to(device)

            # Get logits and loss
            logits, loss, _ = model.forward_logits_loss(images, texts)
            if torch.isfinite(loss):
                val_loss += loss.item()
                n_val += 1

            # Decode: forward without targets gives inference logits
            inf_logits = model.forward(images)
            probs = inf_logits.softmax(-1)
            preds, _ = model.tokenizer.decode(probs)
            all_preds.extend(preds)
            all_refs.extend(texts)

    avg_val_loss = val_loss / max(n_val, 1)
    cer = compute_cer(all_preds, all_refs)
    exact = sum(1 for p, r in zip(all_preds, all_refs) if p.strip() == r.strip()) / max(len(all_preds), 1)
    elapsed = time.time() - epoch_start

    for i in range(min(3, len(all_preds))):
        print(f"  Sample {i}: ref='{all_refs[i][:30]}' pred='{all_preds[i][:30]}'")

    print(f'Epoch {epoch+1}/{EPOCHS} ({elapsed:.0f}s) | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | CER: {cer:.4f} | Exact: {exact:.1%}')

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_cer'].append(cer)
    history['val_exact_match'].append(exact)

    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)

    if cer < best_cer:
        best_cer = cer
        torch.save(model.state_dict(), f'{CHECKPOINT_DIR}/best_model.pt')
        print(f'  New best CER: {best_cer:.4f} -> saved')

print(f'\nTraining complete. Best CER: {best_cer:.4f}')

---
## 6. Results

In [ ]:
# Display results
with open(f'{CHECKPOINT_DIR}/history.json') as f:
    history = json.load(f)

print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'CER':>10} {'Exact':>8}")
print('-' * 50)
for i in range(len(history['train_loss'])):
    print(f"{i+1:>6} {history['train_loss'][i]:>12.4f} {history['val_loss'][i]:>10.4f} {history['val_cer'][i]:>9.2%} {history['val_exact_match'][i]:>7.1%}")

print(f"\nBest CER: {min(history['val_cer']):.4f} (epoch {history['val_cer'].index(min(history['val_cer'])) + 1})")

In [ ]:
# Comparison table
print('\n' + '=' * 70)
print('MODEL COMPARISON — Stage 1 Synthetic Training')
print('=' * 70)
print(f"{'Model':<28} {'Params':>10} {'Best CER':>10} {'Best Exact':>12}")
print('-' * 70)
print(f"{'PARSeq (this run)':<28} {'~24M':>10} {min(history['val_cer']):>9.2%} {max(history['val_exact_match']):>11.1%}")
print(f"{'TrOCR-small unfrozen':<28} {'62M':>10} {'0.06%':>10} {'99.8%':>12}")
print(f"{'TrOCR-small frozen':<28} {'62M':>10} {'0.14%':>10} {'99.5%':>12}")
print(f"{'TrOCR-base unfrozen':<28} {'385M':>10} {'90.4%':>10} {'17.1%':>12}")
print(f"{'CNN+RNN (CTC)':<28} {'~15M':>10} {'66.8%':>10} {'24.8%':>12}")
print(f"{'GPT-4o (zero-shot)':<28} {'~200B':>10} {'98.2%':>10} {'0.0%':>12}")
print('-' * 70)

In [ ]:
# Save results
results = {
    'model': 'PARSeq (ViT + Transformer decoder, permutation LM)',
    'params': sum(p.numel() for p in model.parameters()),
    'best_cer': min(history['val_cer']),
    'best_epoch': history['val_cer'].index(min(history['val_cer'])) + 1,
    'best_exact_match': max(history['val_exact_match']),
    'epochs': len(history['train_loss']),
    'config': {
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'img_size': list(IMG_SIZE),
        'charset_size': len(CHARSET),
        'max_label_len': MAX_LABEL_LEN,
    },
    'history': history,
}

with open(f'{CHECKPOINT_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to: {CHECKPOINT_DIR}/results.json')